# Arc Lamps

Everything so far has worked in pixels. [Chapter 4](04-tracing.ipynb) found where the source falls
on the detector, but nothing in the frame says which wavelength any column holds, and that mapping
has to be measured for the instrument setup. The standard reference is an arc lamp: each line it
emits is a monochromatic image of the slit at a laboratory-known wavelength, a ruler laid across
the detector. Night-sky emission lines, telluric bands and etalons can play the same role, since
`specreduce` needs only a spectrum with lines in it and the wavelengths those lines should have;
this book uses the lamps.

This chapter treats the arc frames as data: how they differ from the science exposure, which part
of the spectrum each lamp covers, and the tilt of their lines, which the next chapter measures and
removes. Working out which catalog wavelength each line has is left to
[Chapter 9](09-line-identification.ipynb).

In [ ]:
import sys

sys.path.append("../src")

import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import simple_norm

from common import read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
nx = obj.data.shape[1]

## What the dataset ships

Three arc exposures, one per lamp (HgAr, Ne and Xe), and the science frame. Their headers
record how each was taken:

In [ ]:
frames = dict(zip(lamps, arcs))
frames["TrES-3b"] = obj

print(f"{'frame':>9}  {'exptime':>7}  {'grism':>7}  {'slit':>21}  {'observed':>20}")
for label, frame in frames.items():
    header = frame.header
    print(
        f"{label:>9}  {header['EXPTIME']:6.1f}s  {header['GRISM']:>7}  "
        f"{header['MASKNAME']:>21}  {header['DATE-OBS'][:19]:>20}"
    )

All four frames were taken through the same grism, `R1000R`, so the arcs disperse light exactly
as the science frame does. Their exposure times differ, 6.2, 2.3 and 8.0 seconds against the
science frame's 12, because each lamp is exposed long enough to bring its brightest lines up
without saturating them. The arcs were also taken on a different night from the science frame.
A spectrograph can flex between nights, and any flexure would be inherited by the wavelength
solution. These are the calibration frames the observatory provided with the service-mode
observations, and we assume the flexure is small enough not to affect the science.

The frames themselves, on the same logarithmic stretch as Chapter 1:

In [ ]:
fig, axs = plt.subplots(3, figsize=(8, 5.4), sharex="all", constrained_layout=True)

for ax, frame, lamp in zip(axs, arcs, lamps):
    ax.imshow(
        frame.data,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(frame.data, stretch="log", vmin=0, vmax=250_000),
    )
    ax.text(0.01, 0.92, lamp, va="top", ha="left", transform=ax.transAxes)

plt.setp(axs, ylabel="Cross-dispersion [pix]")
axs[-1].set(xlim=(0, nx - 1), xlabel="Dispersion [pix]")

# Ticks on the top spine of every panel, so the lean can be read off at both ends of the
# slit; labels only on the topmost panel to keep the stack uncluttered.
for ax in axs:
    ax.tick_params(axis="x", top=True)
axs[0].tick_params(axis="x", labeltop=True);

Each line is a monochromatic image of the slit, so it runs the full height of the frame, and
there is nothing to trace: the science frame's notions of a source and a background do not
apply to an arc. The three lamps divide the detector between them. HgAr holds the blue end,
with its brightest line near column 87, and a cluster between columns 400 and 650; Ne packs a
dense forest of lines between columns 170 and 500; Xe takes over beyond column 640, where the
other two are sparse. Look closely at any line and it is not vertical. It leans and curves
across the slit by an amount that is plain on a frame only 372 rows tall. That is the property
this chapter closes on, and the reason the next chapter exists.

## A different slit

One more difference in that table matters: `MASKNAME`. The arcs went through
`LongSlits1.23Arcsec`; the science exposure through `LongSlits40.0Arcsec`, a slit 32 times
wider.

That is deliberate, and it is the standard setup for transmission spectroscopy. A
monochromatic line is an image of the slit, so a *narrow* slit produces a sharp line whose
centroid can be measured precisely, which is exactly what a wavelength reference needs. The
science exposure has the opposite requirement: a wide slit guarantees that none of the
target's light is clipped by the slit jaws as seeing and guiding wander, at the cost of
spectral resolution. You want sharp arcs and lossless science, and those call for different
slits.


## Collapsing an arc frame to one dimension

To see what each lamp emits, we sum five rows around row 135 into a 1D cut. This is not
`specreduce`'s extraction (Chapter 8), because an arc has no source to extract: its lines fill
the slit. The band is kept narrow for a reason the next section shows.

In [ ]:
ROW, HALF = 135, 2

def row_cut(frame, row=ROW, half=HALF):
    """Sum a narrow band of rows into a 1D cut along the dispersion axis."""
    return frame.data[row - half : row + half + 1].sum(axis=0)

cuts = [row_cut(frame) for frame in arcs]

fig, axs = plt.subplots(3, figsize=(8, 5.0), sharex="all", constrained_layout=True)

for ax, lamp, cut in zip(axs, lamps, cuts):
    ax.plot(cut - np.median(cut), lw=0.6)
    ax.text(0.01, 0.9, lamp, transform=ax.transAxes, va="top", size=8)
    ax.set(yscale="log", ylim=(30, 3e6), ylabel="Signal [e$^-$]")

axs[-1].set(xlim=(0, nx - 1), xlabel="Dispersion [pix]");

The three lamps are complementary: Ne is densest where HgAr is sparse, and Xe takes over in
the red where both thin out. HgAr alone covers the detector unevenly. Its bluest third holds one
very bright line near column 87 and a weak, blended pair near column 160, then nothing until
column 405, partly because the blue end of R1000R is the faint end of its throughput. A solution
fitted to HgAr alone therefore rests on a single blue line, which is why Chapter 10 brings in the
other two lamps.

The lines are about 2.5 pixels FWHM, consistently across the three lamps. A line this narrow
is barely sampled, and the estimate depends on the method: a Gaussian fit to the same lines gives
2.1.

## The lines are not vertical

Every operation in the book so far has assumed that a column of pixels is a single
wavelength. On these frames that is false, and the arc frames are where it is easiest to see,
because an arc line *is* a single wavelength by construction.

We take one window of the Ne frame and cut it at three different heights on the slit.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.0), constrained_layout=True)

window = slice(395, 470)
columns = np.arange(nx)[window]

for row, style in zip((20, 135, 310), ("-", "--", ":")):
    cut = row_cut(arcs[1], row=row)
    ax.plot(
        columns,
        cut[window] / cut[window].max(),
        style,
        lw=1.1,
        label=f"row {row}",
    )

ax.legend(title="cross-dispersion")
ax.set(
    xlabel="Dispersion [pix]",
    ylabel="Normalized signal",
    xlim=(columns[0], columns[-1]),
);

The same lines sit in different columns at different heights on the slit. Between the top and
bottom cuts they shift by several pixels, more than twice the 2.5-pixel line width. A wavelength
solution fitted at one row is therefore wrong at another, and sky measured in rows away from the
target samples slightly different wavelengths from the target's own row. Both have to wait until
the **tilt distortion** has been measured and removed, which is why tilt correction comes next.
How far the lines lean depends on the spectrograph's optics, and a well-aligned instrument may
need no correction at all; Chapter 6 shows how to decide.

## Summary

- **An arc lamp is a ruler**, not a measurement. Its lines are monochromatic images of the
  slit at laboratory-known wavelengths, filling the slit from edge to edge, and they are the
  reference this book uses to put physical units on the dispersion axis. Sky lines, telluric
  bands and etalons can play the same role; `specreduce` only needs lines and their wavelengths.
- **Three lamps, because one is not enough.** Ne is densest where HgAr is sparse and Xe takes
  over in the red. HgAr alone has one bright line near column 87 and nothing usable until column
  405, so a solution fitted to HgAr alone has a single blue anchor, which is why Chapter
  10 brings in the other two lamps.
- **The arcs use a narrower slit than the science frame**, 1.23 against 40 arcseconds, so
  that their lines are sharp.
- **The arcs are from a different night.** Flexure between the nights would be inherited by
  the wavelength solution; these are the service-mode calibrations provided with the data, and
  the book assumes the flexure is small enough not to matter.
- **The lines lean.** This is a **tilt distortion**: a given wavelength falls in columns several
  pixels apart along the slit. Its size depends on the instrument optics; Chapter 6 measures it
  here and shows how to decide whether yours needs correcting.

Next: [Tilt Correction](06-tilt-correction.ipynb), which measures that lean, models it, and
removes it.
